# Data Cleaning

Reads `data.csv`, cleans it, and saves `data_clean.csv`.

In [ ]:
import pandas as pd
import re
import ast

df = pd.read_csv('data.csv')
print(len(df), 'rows loaded')
df.head()

In [ ]:
df = df.drop(columns=['votes'])
df = df.dropna(subset=['title', 'year', 'genre', 'rating'])
print(len(df), 'rows remaining after dropping missing values')

## Year

Year column has 3 formats: `(2022)`, `(2015–2022)`, or `(2018– )` for ongoing shows. We split it into `start_year` and `end_year`.

In [ ]:
def get_years(text):
    text = str(text)
    years = re.findall(r'\d{4}', text)

    if len(years) == 0:
        return None, None
    if len(years) >= 2:
        return int(years[0]), int(years[1])

    start = int(years[0])
    if '-' in text or '–' in text:
        return start, 2024
    return start, start


start_years = []
end_years = []

for val in df['year']:
    s, e = get_years(val)
    start_years.append(s)
    end_years.append(e)

df['start_year'] = start_years
df['end_year'] = end_years

df = df.dropna(subset=['start_year', 'end_year'])
df['start_year'] = df['start_year'].astype(int)
df['end_year'] = df['end_year'].astype(int)

df = df.drop(columns=['year'])
df[['title', 'start_year', 'end_year']].head()

## Duration, Certificate, Genre

In [ ]:
# duration is like '45 min', take the number before the space
def get_minutes(text):
    try:
        return int(str(text).split()[0])
    except:
        return None

df['duration_min'] = df['duration'].apply(get_minutes)
median_dur = int(df['duration_min'].median())
df['duration_min'] = df['duration_min'].fillna(median_dur).astype(int)
df = df.drop(columns=['duration'])

df['certificate'] = df['certificate'].fillna('Unrated')

# strip extra spaces around each genre name
def clean_genre(text):
    return ', '.join([g.strip() for g in text.split(',')])

df['genre'] = df['genre'].apply(clean_genre)

## Stars

Some rows have `| Stars:` before the actor names (others don't). We keep only actor names.

In [ ]:
def clean_stars(text):
    items = ast.literal_eval(str(text))

    for i, item in enumerate(items):
        if 'Stars:' in item or 'Star:' in item:
            items = items[i + 1:]
            break

    names = [name.strip(', ') for name in items if name.strip(', ') not in ('', '|')]
    return ', '.join(names)


df['stars'] = df['stars'].apply(clean_stars)
df[['title', 'stars']].head()

In [ ]:
df = df[['title', 'start_year', 'end_year', 'certificate', 'duration_min', 'rating', 'genre', 'stars', 'description']]

df.to_csv('data_clean.csv', index=False)
print('saved data_clean.csv with', len(df), 'rows')
df.head()